In [ ]:
# cell 1
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
# cell 2
# File paths and constants
SURVEY_FILE = r"..\Data\raw data\australia.csv"
OUTPUT_DIR = Path(r"..\Data\processed_data")
OUTPUT_DIR.mkdir(exist_ok=True)

MISSING_RATE_THRESHOLD = 0.21

CONSENT_START = "2021-02-10"
CONSENT_END = "2021-10-18"

MIN_VALID_MASK_ITEMS = 2
MIN_VALID_GENERAL_ITEMS = 9
MIN_VALID_SOCIAL_AVOIDANCE_ITEMS = 4

In [ ]:
# cell 3
# Load raw survey data

survey_raw = pd.read_csv(
    SURVEY_FILE,
    na_values=[" ", "__NA__"],
    keep_default_na=True,
    low_memory=False
)

survey_df = survey_raw.copy()

print("Raw survey shape:", survey_df.shape)

Raw survey shape: (53833, 513)


In [ ]:
# cell 4
# Clean survey date

survey_df["endtime"] = pd.to_datetime(
    survey_df["endtime"].astype(str).str.split().str[0],
    format="%d/%m/%Y",
    errors="coerce"
)

survey_df = survey_df.dropna(subset=["endtime"]).copy()

print("Shape after removing missing endtime:", survey_df.shape)
print("Date range:", survey_df["endtime"].min(), "to", survey_df["endtime"].max())

Shape after removing missing endtime: (53833, 513)
Date range: 2020-04-01 00:00:00 to 2022-03-28 00:00:00


In [ ]:
# cell 5
# Check state values

if "state" in survey_df.columns:
    print("State values:")
    print(survey_df["state"].value_counts(dropna=False))
else:
    raise ValueError("Column 'state' is missing from the survey data.")

State values:
state
New South Wales                 16473
Victoria                        13899
Queensland                      10963
Western Australia                5093
South Australia                  5060
Tasmania                         1124
Australian Capital Territory      859
Northern Territory                362
Name: count, dtype: int64


In [ ]:
# cell 6
# Missing value summary before dropping high-missing columns

missing_rate = survey_df.isna().mean().sort_values(ascending=False)

missing_summary = pd.DataFrame({
    "variable": missing_rate.index,
    "missing_rate": missing_rate.values
})

missing_summary.to_csv(
    OUTPUT_DIR / "survey_missing_value_summary_before_drop.csv",
    index=False
)

high_missing_cols = missing_rate[missing_rate > MISSING_RATE_THRESHOLD].index.tolist()

print("Number of columns with high missing rate:", len(high_missing_cols))
print(high_missing_cols[:30])

Number of columns with high missing rate: 460
['V3_child2_4_other', 'V3_baby_other', 'm6_other', 'V3_baby', 'V3_me_other', 'V3_child5_17_other', 'm8_other', 'V3_adult18_other', 'ct3_other', 'ct5_other', 'V3_child2_4', 'V3_child5_17', 'v3_open', 'ct6_other', 'm4_other', 'm12_other', 'ct1b_other', 'V3_me', 'V4_other', 'm6_7', 'm6_6', 'm6_5', 'm6_4', 'm6_8', 'm6_96', 'm6_2', 'm6_1', 'm6_3', 'V3_adult18', 'v3_other']


In [ ]:
# cell 7
# Drop columns with high missing rate

# Keep key columns even if their missing rate is high
protected_cols = [
    "RecordNo",
    "endtime",
    "state"
]

high_missing_cols = [
    col for col in high_missing_cols
    if col not in protected_cols
]

survey_df = survey_df.drop(columns=high_missing_cols, errors="ignore").copy()

print("Shape after dropping high-missing columns:", survey_df.shape)

Shape after dropping high-missing columns: (53833, 53)


In [ ]:
# cell 8
# Handle consent-period missing values for PHQ and comorbidity items

consent_mask = (
    (survey_df["endtime"] >= pd.to_datetime(CONSENT_START)) &
    (survey_df["endtime"] <= pd.to_datetime(CONSENT_END))
)

phq_cols = [f"PHQ4_{i}" for i in range(1, 5)]

d1_health_cols = (
    [f"d1_health_{i}" for i in range(1, 14)] +
    [f"d1_health_{i}" for i in range(98, 100)]
)

for col in phq_cols:
    if col in survey_df.columns:
        survey_df.loc[consent_mask, col] = (
            survey_df.loc[consent_mask, col].fillna("N/A")
        )

for col in d1_health_cols:
    if col in survey_df.columns:
        survey_df.loc[consent_mask, col] = (
            survey_df.loc[consent_mask, col].fillna("N/A")
        )

In [ ]:
# cell 9
# Remove remaining missing values after dropping high-missing columns

print("Shape before complete-case deletion:", survey_df.shape)

missing_before_dropna = (
    survey_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_before_dropna = missing_before_dropna[missing_before_dropna > 0]

print("Columns with remaining missing values before dropna:")
print(missing_before_dropna)

survey_df = survey_df.dropna().copy()

print("Shape after complete-case deletion:", survey_df.shape)

Shape before complete-case deletion: (53833, 53)
Columns with remaining missing values before dropna:
i12_health_23     10029
r1_2               9023
r1_1               9023
i12_health_25      9023
i12_health_22      9023
WCRex1             7014
PHQ4_2             4395
PHQ4_4             4376
PHQ4_1             4373
PHQ4_3             4371
cantril_ladder     3985
WCRex2             2981
d1_health_4        2593
d1_health_99       2593
d1_health_98       2593
d1_health_13       2593
d1_health_12       2593
d1_health_11       2593
d1_health_10       2593
d1_health_9        2593
d1_health_8        2593
d1_health_6        2593
d1_health_5        2593
d1_health_7        2593
d1_health_3        2593
d1_health_1        2593
d1_health_2        2593
i2_health          2002
i12_health_2       1006
i12_health_11      1006
i9_health          1001
i11_health         1001
dtype: int64
Shape after complete-case deletion: (41125, 53)


In [ ]:
# cell 10
# Encode behavioural frequency variables

frequency_map = {
    "Always": 5,
    "Frequently": 4,
    "Sometimes": 3,
    "Rarely": 2,
    "Not at all": 1
}

i12_health_cols = [
    col for col in survey_df.columns
    if col.startswith("i12_health_")
]

print("Number of i12_health columns found:", len(i12_health_cols))
print(i12_health_cols)

for col in i12_health_cols:
    survey_df[col] = survey_df[col].map(frequency_map)


Number of i12_health columns found: 17
['i12_health_1', 'i12_health_2', 'i12_health_3', 'i12_health_4', 'i12_health_5', 'i12_health_6', 'i12_health_7', 'i12_health_8', 'i12_health_11', 'i12_health_12', 'i12_health_13', 'i12_health_14', 'i12_health_15', 'i12_health_16', 'i12_health_22', 'i12_health_23', 'i12_health_25']


In [ ]:
# cell 11
# Define behavioural outcome items

mask_items = [
    "i12_health_1",
    "i12_health_22",
    "i12_health_23",
    "i12_health_25"
]

social_avoidance_items = [
    "i12_health_6",
    "i12_health_8",
    "i12_health_11",
    "i12_health_12",
    "i12_health_13",
    "i12_health_14",
    "i12_health_15"
]

mask_items = [col for col in mask_items if col in survey_df.columns]
social_avoidance_items = [
    col for col in social_avoidance_items
    if col in survey_df.columns
]

general_protective_items = i12_health_cols.copy()

print("Mask items:", mask_items)
print("Social avoidance items:", social_avoidance_items)
print("General protective item count:", len(general_protective_items))

Mask items: ['i12_health_1', 'i12_health_22', 'i12_health_23', 'i12_health_25']
Social avoidance items: ['i12_health_6', 'i12_health_8', 'i12_health_11', 'i12_health_12', 'i12_health_13', 'i12_health_14', 'i12_health_15']
General protective item count: 17


In [ ]:
# cell 12
# Helper function for constructing behavioural outcomes

def construct_median_outcome(
    df,
    item_cols,
    scale_col,
    binary_col,
    valid_count_col,
    min_valid_items,
    threshold=4
):
    df[valid_count_col] = df[item_cols].notna().sum(axis=1)

    df[scale_col] = df[item_cols].median(axis=1, skipna=True)

    df.loc[df[valid_count_col] < min_valid_items, scale_col] = np.nan

    df[binary_col] = pd.NA

    df.loc[df[scale_col] >= threshold, binary_col] = "Yes"
    df.loc[df[scale_col] < threshold, binary_col] = "No"

    return df

In [ ]:
# cell 13
# Construct face mask behaviour outcome

survey_df = construct_median_outcome(
    df=survey_df,
    item_cols=mask_items,
    scale_col="face_mask_behaviour_scale",
    binary_col="face_mask_behaviour_binary",
    valid_count_col="face_mask_valid_count",
    min_valid_items=MIN_VALID_MASK_ITEMS,
    threshold=4
)

print(survey_df["face_mask_behaviour_binary"].value_counts(dropna=False))

face_mask_behaviour_binary
Yes    22157
No     18968
Name: count, dtype: int64


In [ ]:
# cell 14
# Construct general protective behaviour outcome

survey_df = construct_median_outcome(
    df=survey_df,
    item_cols=general_protective_items,
    scale_col="protective_behaviour_scale",
    binary_col="protective_behaviour_binary",
    valid_count_col="protective_behaviour_valid_count",
    min_valid_items=MIN_VALID_GENERAL_ITEMS,
    threshold=4
)

print(survey_df["protective_behaviour_binary"].value_counts(dropna=False))

protective_behaviour_binary
Yes    26686
No     14439
Name: count, dtype: int64


In [ ]:
# cell 15
# Construct social avoidance behaviour outcome

survey_df = construct_median_outcome(
    df=survey_df,
    item_cols=social_avoidance_items,
    scale_col="social_avoidance_scale",
    binary_col="social_avoidance_binary",
    valid_count_col="social_avoidance_valid_count",
    min_valid_items=MIN_VALID_SOCIAL_AVOIDANCE_ITEMS,
    threshold=4
)

print(survey_df["social_avoidance_binary"].value_counts(dropna=False))

social_avoidance_binary
Yes    24202
No     16923
Name: count, dtype: int64


In [ ]:
# cell 16
# Construct non-mask protective behaviour scale

non_mask_items = [
    col for col in general_protective_items
    if col not in mask_items
]

survey_df["protective_behaviour_nomask_valid_count"] = (
    survey_df[non_mask_items].notna().sum(axis=1)
)

survey_df["protective_behaviour_nomask_scale"] = (
    survey_df[non_mask_items].median(axis=1, skipna=True)
)

survey_df.loc[
    survey_df["protective_behaviour_nomask_valid_count"] < 6,
    "protective_behaviour_nomask_scale"
] = np.nan

In [ ]:
# cell 17
# Encode risk perception variables

agreement_map = {
    "7 - Agree": 7,
    "6": 6,
    "5": 5,
    "4": 4,
    "3": 3,
    "2": 2,
    "1 – Disagree": 1
}

for col in ["r1_1", "r1_2"]:
    if col in survey_df.columns:
        survey_df[col] = survey_df[col].replace(agreement_map)
        survey_df[col] = pd.to_numeric(survey_df[col], errors="coerce")

C:\Users\dukai\AppData\Local\Temp\ipykernel_22824\703094212.py:15: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  survey_df[col] = survey_df[col].replace(agreement_map)


In [ ]:
# cell 18
# Clean household size

def convert_household_size(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value in [str(i) for i in range(1, 8)]:
        return int(value)

    if value == "8 or more":
        return 8

    if value in ["Prefer not to say", "Don't know", "nan"]:
        return np.nan

    return np.nan


if "household_size" in survey_df.columns:
    survey_df["household_size"] = survey_df["household_size"].apply(
        convert_household_size
    )

In [ ]:
# cell 19
# Construct comorbidity indicator

d1_cols = [
    col for col in survey_df.columns
    if col.startswith("d1_")
]

survey_df["d1_comorbidities"] = np.nan

if len(d1_cols) > 0:
    survey_df["d1_comorbidities"] = "Yes"

    if "d1_health_99" in survey_df.columns:
        survey_df.loc[
            survey_df["d1_health_99"] == "Yes",
            "d1_comorbidities"
        ] = "No"

        survey_df.loc[
            survey_df["d1_health_99"] == "N/A",
            "d1_comorbidities"
        ] = "N/A"

    if "d1_health_98" in survey_df.columns:
        survey_df.loc[
            survey_df["d1_health_98"] == "Yes",
            "d1_comorbidities"
        ] = "Prefer_not_to_say"

In [ ]:
# cell 20
# Construct survey time variable

start_date = survey_df["endtime"].min()

survey_df["week_number"] = (
    (survey_df["endtime"] - start_date).dt.days // 14
) + 1

print("Week number range:", survey_df["week_number"].min(), "to", survey_df["week_number"].max())

Week number range: 1 to 44


In [ ]:
# cell 21
# Drop technical columns that should not be predictors

drop_cols = [
    "qweek",
    "weight"
]

survey_df = survey_df.drop(columns=drop_cols, errors="ignore")

In [ ]:
# cell 22
# Save a detailed cleaned survey dataset

output_file = OUTPUT_DIR / "cleaned_survey_base.csv"

survey_df.to_csv(output_file, index=False)

print("Saved:", output_file)
print("Cleaned survey shape:", survey_df.shape)

Saved: ..\Data\processed_data\cleaned_survey_base.csv
Cleaned survey shape: (41125, 64)


In [ ]:
# cell 23
# Output outcome summary

outcome_summary = pd.DataFrame({
    "outcome": [
        "face_mask_behaviour_binary",
        "protective_behaviour_binary",
        "social_avoidance_binary"
    ],
    "yes_count": [
        (survey_df["face_mask_behaviour_binary"] == "Yes").sum(),
        (survey_df["protective_behaviour_binary"] == "Yes").sum(),
        (survey_df["social_avoidance_binary"] == "Yes").sum()
    ],
    "no_count": [
        (survey_df["face_mask_behaviour_binary"] == "No").sum(),
        (survey_df["protective_behaviour_binary"] == "No").sum(),
        (survey_df["social_avoidance_binary"] == "No").sum()
    ],
    "missing_count": [
        survey_df["face_mask_behaviour_binary"].isna().sum(),
        survey_df["protective_behaviour_binary"].isna().sum(),
        survey_df["social_avoidance_binary"].isna().sum()
    ]
})

outcome_summary.to_csv(
    OUTPUT_DIR / "survey_outcome_summary.csv",
    index=False
)

outcome_summary

,outcome,yes_count,no_count,missing_count
0,face_mask_behaviour_binary,22157,18968,0
1,protective_behaviour_binary,26686,14439,0
2,social_avoidance_binary,24202,16923,0
